# P2b: the selector on real and simulated data
Observed only. Thresholds not tuned on real data.

In [1]:
# A. Selector on real data (full sample, nothing is tuned here)
# GOAL: run select_and_fit once per FX pair and show what it chose and why.
#
# How to read the table (one row per pair):
#   mean_m / mean_reason = AR order the mean rule picked (0 = constant mean) and its evidence
#   mean_LB10_p          = Ljung-Box p of the mean-model residuals; > 0.05 means the mean
#                          model removed the autocorrelation, < 0.05 means some is left over
#   start / final        = variance order (p, q) the ACF/PACF rule proposed / the order kept
#                          after the gate and escalation (p = shock lags, q = variance lags)
#   persistence          = sum of alpha and beta; close to 1 = shocks to volatility die out slowly
#   gate_pass            = squared standardised residuals look like white noise
#                          (the model absorbed the volatility clustering)
#   models_fitted        = how many GARCH fits were needed (cost of the selection)
#   flagged              = True when the gate still failed after all escalation steps

# --- Imports and project path ---------------------------------------------------------
import sys                      # lets us edit Python's module search path
from pathlib import Path        # object-style file paths

# The notebook lives in notebooks/, but our code lives in src/ one folder up.
# If the current folder has no src/, step up one level to find the project root.
root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(root))   # put the project root first so 'from src...' works

import pandas as pd             # tables

from src.data import load_config, load_fx, to_log_returns   # config, FRED prices, returns
from src.selector import select_and_fit                     # the whole selection pipeline

# Display options so the wide table stays readable.
pd.set_option('display.width', 250)         # allow wide output before wrapping
pd.set_option('display.max_colwidth', 70)   # cut long reason strings at 70 characters

cfg = load_config()   # every threshold, lag and seed comes from config.yaml


# --- Helper: turn one result dict into one table row ----------------------------------
def summary_row(name, n, result):
    """Flatten one select_and_fit result into a table row (presentation only)."""
    log = result['escalation_log']   # one entry per fitted model, in the order tried

    return {
        'pair': name,
        'n': n,                                           # number of return observations
        'mean_m': result['mean_spec']['m'],               # AR order of the mean (0 = constant)
        'mean_reason': result['mean_spec']['reason'],     # why the mean rule chose it
        'mean_LB10_p': result['mean_resid_lb_p'],         # autocorrelation left after the mean fit
        'start': log[0]['order'],                         # first fit = the order the rule proposed
        'final': result['order'],                         # order of the model finally returned
        'dist': result['dist'],                           # 'normal' or 't' (after escalation)
        'persistence': result['persistence'],             # NaN means no valid fit was found
        'gate_pass': result['gate']['passed'],
        'models_fitted': result['models_fitted'],
        'flagged': result['flagged'],
        'log_len': len(log),                              # equals models_fitted (sanity check)
    }


# --- Run the selector on each real pair ------------------------------------------------
real_rows = []

for pair in cfg['pairs']:
    # Load the cached FRED prices (downloads only if the CSV is missing).
    prices = load_fx(pair, cfg['start'], cfg.get('end'), cfg)

    # Daily log returns x100, roughly the percent change per day.
    returns = to_log_returns(prices)

    # Full-sample run. The selector only sees this series, so there is no look-ahead.
    result = select_and_fit(returns, cfg)

    real_rows.append(summary_row(pair, len(returns), result))

# Last line of a cell is displayed automatically: one row per pair.
pd.DataFrame(real_rows).set_index('pair')

,n,mean_m,mean_reason,mean_LB10_p,start,final,dist,persistence,gate_pass,models_fitted,flagged,log_len
pair,,,,,,,,,,,,
DEXUSEU,4190,0,"constant: no ACF/PACF spike, LB(10) p=0.334",0.333613,"(1, 1)","(1, 1)",normal,0.994316,True,1,False,1
DEXJPUS,4190,0,"constant: no ACF/PACF spike, LB(10) p=0.179",0.178890,"(1, 1)","(1, 1)",normal,0.991977,True,1,False,1
DEXINUS,4189,2,"AR(2): ACF spikes [1, 2, 4, 5], PACF spikes [1, 2, 4, 5], LB(10) p...",0.001462,"(1, 1)","(1, 1)",normal,NaN,False,4,True,4


In [2]:
# B. Selector on simulated data where the true order is known
# GOAL: build the same table as cell A for a GARCH(1,1) and an ARCH(1) series, to see how the
# selector behaves when the truth is known. One run each, so this is an illustration, not a
# recovery rate (that is the P3 experiment). Parameters and seed come from config.yaml (sim.demo).
# Reading hint: the true mean is constant, so any 'AR(1)' in mean_reason is a false positive
# from the rule (D-016), and the true variance order is (1, 1) / (1, 0) respectively.
from src.simulate import sim_arch1, sim_garch11   # our hand-rolled simulators

# --- Read the simulation settings from config -----------------------------------------
sim = cfg['sim']        # seed, burn-in, test and demo settings
demo = sim['demo']      # n and the true parameters for this notebook
g, a = demo['garch11'], demo['arch1']   # true GARCH(1,1) and ARCH(1) parameters

# --- Simulate the two series (same seed, so the table is reproducible) ----------------
sim_series = {
    # h_t = omega + alpha * e_{t-1}^2 + beta * h_{t-1}
    'sim GARCH(1,1)': sim_garch11(demo['n'], g['omega'], g['alpha'], g['beta'], sim['seed'], sim['burn']),
    # ARCH(1): same recursion with beta = 0 (no memory of past variance)
    'sim ARCH(1)': sim_arch1(demo['n'], a['omega'], a['alpha'], sim['seed'], sim['burn']),
}

# --- Run the selector on each simulated series ----------------------------------------
sim_rows = []

for name, series in sim_series.items():
    result = select_and_fit(series, cfg)   # same pipeline as for the real data
    sim_rows.append(summary_row(name, len(series), result))   # same helper as cell A

# Displayed automatically: one row per simulated series.
pd.DataFrame(sim_rows).set_index('pair')

,n,mean_m,mean_reason,mean_LB10_p,start,final,dist,persistence,gate_pass,models_fitted,flagged,log_len
pair,,,,,,,,,,,,
"sim GARCH(1,1)",2000,1,"AR(1): ACF spikes [], PACF spikes [], LB(10) p=0.033",0.043161,"(1, 1)","(1, 1)",normal,0.941723,True,1,False,1
sim ARCH(1),2000,1,"AR(1): ACF spikes [1], PACF spikes [1], LB(10) p=0.0723",0.269404,"(1, 1)","(1, 1)",normal,0.430397,True,1,False,1


## P3a: calibration on simulated data only
Criteria are fixed in decision.md D-018 and the addendum D-021 before these numbers exist. Real data is not used below.

D-022 adds a cut-off-read grid (cell s06): cutoff C0/C1 x fade F2/F3/F4 on arch1, arch2, garch11. Criterion fixed in D-022 before any of its numbers.

In [3]:
# C. Calibration of two rules on SIMULATED data only (D-018, extended by D-021)
# GOAL: for each candidate setting of the fade tolerance (F0..F4) and the mean trigger
# (M0/M1), run the selector R times on series with a KNOWN truth and count how often it is right.
# Nothing here touches real data, and only calibration seeds are used (never eval seeds).
# Every candidate sees the same seeds, so candidates are paired replication by replication.
#
# How to read the output:
#   Table 1 (fade): exact-order recovery in [0, 1]. Columns arch1 / garch11 = share of
#     replications where the selector returned exactly the true (p, q); score = mean of the
#     two; garch11_ge_090 = True when GARCH(1,1) recovery meets the 0.90 floor (D-021).
#   Table 2 (mean trigger): any_ar_<model> = share of series given ANY AR mean (m > 0).
#     For iid, arch1, garch11 the truth is m = 0, so this is a false-AR rate (lower is better).
#     For ar1_garch11 the truth is m = 1, so it is a detection rate (higher is better).
#     ar1_detect = share of ar1_garch11 series given exactly AR(1).
#   Table 3: every number above plus cost (mean_models_fitted) and failure shares.
#   Table 4 (orders): per fade candidate and true model, how often each (start order, final
#     order) pair occurred. start = what the ACF/PACF rule proposed, final = what survived the
#     gate and escalation. Also saved to results/calibration_orders.csv.
# The winners are NOT picked here; the criteria are fixed in decision.md (D-018, D-021) and a
# separate step freezes them after you paste these tables.

from src.simulate import calibration_table, summarise_calibration   # harness lives in src/

# --- Run the whole calibration (R series per candidate and true model, joblib in parallel) ---
table, orders = calibration_table(cfg)

# --- Save both long tables ----------------------------------------------------------------
results_dir = root / cfg['paths']['results_dir']   # results/ at the project root
results_dir.mkdir(exist_ok=True)                    # create it the first time
table.to_csv(results_dir / 'calibration.csv', index=False)
orders.to_csv(results_dir / 'calibration_orders.csv', index=False)

# --- Short comparison tables used by the selection criteria ------------------------------
fade_df, mean_df = summarise_calibration(table)

print('Table 1: fade candidates, exact-order recovery (higher is better)')
print(fade_df.round(3).to_string())

print('\nTable 2: mean-trigger candidates, share given any AR mean (false-AR for m=0 truths)')
print(mean_df.round(3).to_string())

print(f"\nCalibration: R={cfg['calibration']['R']}, n={cfg['calibration']['n']}, "
      f"seeds {cfg['calibration']['seed_base']}..{cfg['calibration']['seed_base'] + cfg['calibration']['R'] - 1}")

# Displayed automatically: Table 3, the full numbers behind the two tables above.
# (Table 4, the order counts, is in results/calibration_orders.csv.)
table.round(3)

Table 1: fade candidates, exact-order recovery (higher is better)
true_model  arch1  garch11  score  garch11_ge_090
candidate                                        
F0           0.39     0.97  0.680            True
F1           0.52     0.95  0.735            True
F2           0.61     0.95  0.780            True
F3           0.65     0.94  0.795            True
F4           0.66     0.93  0.795            True

Table 2: mean-trigger candidates, share given any AR mean (false-AR for m=0 truths)
           any_ar_ar1_garch11  any_ar_arch1  any_ar_garch11  any_ar_iid  ar1_detect
candidate                                                                          
M0                        1.0          0.39            0.44        0.16        0.93
M1                        1.0          0.17            0.18        0.05        0.93

Calibration: R=100, n=1000, seeds 1000000..1000099


,rule,candidate,true_model,n,R,exact_order,any_ar_rate,m_correct,mean_models_fitted,flagged_rate,no_valid_fit_rate
0,fade,F0,arch1,1000,100,0.39,0.39,0.61,1.12,0.04,0.0
1,fade,F0,garch11,1000,100,0.97,0.44,0.56,1.17,0.06,0.0
2,fade,F1,arch1,1000,100,0.52,0.39,0.61,1.12,0.04,0.0
3,fade,F1,garch11,1000,100,0.95,0.44,0.56,1.17,0.06,0.0
4,fade,F2,arch1,1000,100,0.61,0.39,0.61,1.12,0.04,0.0
5,fade,F2,garch11,1000,100,0.95,0.44,0.56,1.19,0.06,0.0
6,fade,F3,arch1,1000,100,0.65,0.39,0.61,1.12,0.04,0.0
7,fade,F3,garch11,1000,100,0.94,0.44,0.56,1.20,0.06,0.0
8,fade,F4,arch1,1000,100,0.66,0.39,0.61,1.12,0.04,0.0
9,fade,F4,garch11,1000,100,0.93,0.44,0.56,1.22,0.06,0.0


In [4]:
# D. PACF cut-off read: calibration round on SIMULATED data only (D-022)

# GOAL: compare two ways of reading the ARCH order p off the PACF of squared residuals
# (C0 = last significant lag, C1 = unbroken run from lag 1) crossed with three fade
# tolerances (F2, F3, F4), and see which recovers the true order most often.
# Nothing here touches real data, and only calibration seeds are used (never eval seeds).
# Every (cutoff, fade) pair sees the SAME seeds, so the comparison is paired.
#
# How to read the output:
#   Table 5: one row per (cutoff, fade). Columns arch1 / arch2 / garch11 = share of
#     replications where the selector returned exactly the true (p, q); score = their mean;
#     garch11_ge_090 = the 0.90 floor on GARCH(1,1); gain_vs_C0F3 = score minus the (C0, F3)
#     score; clears_margin = floor holds AND the gain is at least 0.03 (the D-022 margin).
#   results/calibration_cutoff.csv: cutoff, fade, true_model, exact_order, flagged_rate.
#   results/calibration_cutoff_orders.csv: for each cell, how often each (start order,
#     final order) pair occurred. start = what the rule proposed, final = after the gate.
# The winner is NOT picked here; the criterion is fixed in decision.md (D-022) and a
# separate step freezes it after you paste these tables.

from src.simulate import cutoff_calibration_table, summarise_cutoff   # harness lives in src/


# --- Run the grid (R series per cell, joblib in parallel; same seeds for every cell) ---
cut_table, cut_orders = cutoff_calibration_table(cfg)


# --- Save both long tables (results/ was created by the previous cell) ----------------
results_dir = root / cfg['paths']['results_dir']
results_dir.mkdir(exist_ok=True)
cut_table.to_csv(results_dir / 'calibration_cutoff.csv', index=False)
cut_orders.to_csv(results_dir / 'calibration_cutoff_orders.csv', index=False)


# --- Short comparison table used by the D-022 criterion -------------------------------
cut_summary = summarise_cutoff(cut_table, cfg)

print('Table 5: cut-off x fade, exact-order recovery (higher is better)')
print(cut_summary.round(3).to_string())

print(f"\nGrid: R={cfg['calibration']['R']}, n={cfg['calibration']['n']}, "
      f"seeds {cfg['calibration']['seed_base']}..{cfg['calibration']['seed_base'] + cfg['calibration']['R'] - 1}")

Table 5: cut-off x fade, exact-order recovery (higher is better)
true_model   arch1  arch2  garch11  score  garch11_ge_090  gain_vs_C0F3  clears_margin
cutoff fade                                                                           
C0     F2     0.61   0.45     0.95  0.670            True        -0.013          False
       F3     0.65   0.46     0.94  0.683            True         0.000          False
       F4     0.66   0.46     0.93  0.683            True         0.000          False
C1     F2     0.61   0.45     0.95  0.670            True        -0.013          False
       F3     0.65   0.46     0.94  0.683            True         0.000          False
       F4     0.66   0.46     0.93  0.683            True         0.000          False

Grid: R=100, n=1000, seeds 1000000..1000099


Observed only. Thresholds not tuned on real data.

## Frozen rules (D-023, 2026-10-08)

The selector rules are frozen: cut-off rule **C0** (last significant PACF lag), fade tolerance **F3** (up to 3 spikes), mean trigger **M1** (Ljung-Box only), `persistence_tol = 1e-6`. Values live in `config.yaml` (`frozen` key) and are pinned by `tests/test_frozen.py`.

- The cells above reproduce the calibration with the other candidates (F0-F4, M0, C1). Those are calibration only.
- Evidence: `results/calibration.csv`, `results/calibration_cutoff.csv`. (C0, F3) ties F4 at mean exact-order recovery 0.683 and wins on fewer spikes; C1 is identical to C0.
- Known limits: exact recovery ARCH(1) 0.65, ARCH(2) 0.46, GARCH(1,1) 0.94; M1 false-AR 0.05 (iid), 0.17 (ARCH(1)), 0.18 (GARCH(1,1)).
- Change policy: any rule change needs a decision entry, a calibration rerun and the eval-seed recovery rerun. Real data is never used to tune.